# Epistemic FactKG — Hyperparameter Search (Colab GPU)

**Runtime → Change runtime type → T4 GPU** before running.

Models: v3-nli → v2-hgnn → v1-hgnn → baseline  ·  30 trials each  
Pre-requisite: `colab_upload.zip` uploaded to Google Drive root  
(`just colab-prep` on your local machine generates it).

In [ ]:
# ── Cell 1: Mount Drive + extract project ────────────────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib, os

ZIP     = pathlib.Path('/content/drive/MyDrive/colab_upload.zip')
PROJECT = pathlib.Path('/content/epistemic-factkg')

if not PROJECT.exists():
    print('Extracting...')
    PROJECT.mkdir(parents=True)
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(PROJECT)
    print('Done.')
else:
    print('Already extracted.')

os.chdir(PROJECT)
print('cwd:', os.getcwd())

In [ ]:
# ── Cell 2: Install uv + Python 3.14 + dependencies ──────────
!pip install uv -q
!uv python install 3.14
!uv sync --python 3.14 -q

# Verify GPU
!uv run python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# ── Cell 3: Verify required data files ───────────────────────
import pathlib

required = [
    'out/data/training/epistemic_factkg_training.jsonl',
    'out/data/splits/train_indices.json',
    'out/data/splits/val_indices.json',
    'out/data/splits/test_indices.json',
    'data/registry/source_trust_registry.jsonl',
]
all_ok = True
for p in required:
    ok = pathlib.Path(p).exists()
    print(('✅' if ok else '❌'), p)
    if not ok: all_ok = False

for d in ['out/model/graphs', 'configs/hparams']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

assert all_ok, 'Some data files are missing — re-run just colab-prep and re-upload.'

In [ ]:
# ── Cell 4: v3-nli hparam search (downloads NLI model ~250 MB on first run) ────────────────
!uv run python -m src.pipeline.model.hparam_search \\
    --model v3-nli \\
    --jsonl out/data/training/epistemic_factkg_training.jsonl \\
    --splits-dir out/data/splits \\
    --n-trials 30 \\
    --batch-size 32

In [ ]:
# ── Cell 5: v2-hgnn hparam search ────────────────
!uv run python -m src.pipeline.model.hparam_search \\
    --model v2-hgnn \\
    --jsonl out/data/training/epistemic_factkg_training.jsonl \\
    --splits-dir out/data/splits \\
    --n-trials 30 \\
    --batch-size 32

In [ ]:
# ── Cell 6: Copy graph cache → v1-hgnn + baseline ──
# v1-hgnn and baseline use identical GraphConfig.v1() with no NLI.
# Reusing the v2-hgnn cache skips ~10 min of graph building per model.
import shutil, pathlib

src = pathlib.Path('out/model/graphs/split_cache_v2-hgnn.pkl')
for m in ['v1-hgnn', 'baseline']:
    dst = pathlib.Path(f'out/model/graphs/split_cache_{m}.pkl')
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f'Copied cache → {dst}')
    elif dst.exists():
        print(f'Cache exists: {dst}')
    else:
        print(f'WARN: source cache not found — graph will be rebuilt')

In [ ]:
# ── Cell 6: v1-hgnn hparam search ────────────────
!uv run python -m src.pipeline.model.hparam_search \\
    --model v1-hgnn \\
    --jsonl out/data/training/epistemic_factkg_training.jsonl \\
    --splits-dir out/data/splits \\
    --n-trials 30 \\
    --batch-size 32

In [ ]:
# ── Cell 7: baseline hparam search ────────────────
!uv run python -m src.pipeline.model.hparam_search \\
    --model baseline \\
    --jsonl out/data/training/epistemic_factkg_training.jsonl \\
    --splits-dir out/data/splits \\
    --n-trials 30 \\
    --batch-size 32

In [ ]:
# ── Cell 9: Results summary ─────────────────
import json, pathlib

for model in ['v3-nli', 'v2-hgnn', 'v1-hgnn', 'baseline']:
    p = pathlib.Path(f'configs/hparams/{model}_best_hparams.json')
    if p.exists():
        d = json.loads(p.read_text())
        print(f'\n── {model}  val_loss={d.get("_best_val_loss")}')
        for k, v in d.items():
            if not k.startswith('_'):
                print(f'  {k}: {v}')
    else:
        print(f'❌  {model}: not found')

In [ ]:
# ── Cell 10: Save hparam files back to Drive ──
import shutil, pathlib

OUT = pathlib.Path('/content/drive/MyDrive/epistemic-factkg-hparams')
OUT.mkdir(exist_ok=True)

for f in pathlib.Path('configs/hparams').glob('*_best_hparams.json'):
    shutil.copy(f, OUT / f.name)
    print(f'Saved → {OUT / f.name}')

print('\nDone. Copy these files back to your local configs/hparams/ folder.')